# 09 · Cohort sensitivity with a documented-testosterone requirement

A self-contained re-run of the cohort-sensitivity work (`02_univariate.ipynb` + `02a_cohort_comparison.ipynb`
+ the Figure 3 PSA/testosterone forest) with **one more pre-castrate filter** stacked on the existing
`+noprecastrate` exclusion:

> A patient must have a **documented testosterone at or above the castrate level (>= 50 ng/dL) in the 6 months
> strictly before their ADT start date.**

The notebook builds everything it needs. It does **not** require any change to `01`/`02`/`05`, to the
`compass_pipeline` package, or to the R figure code:

| step | what it does |
|---|---|
| 1 | Derives the new exclusion MRN list from `longitudinal_prediction_data_adt.csv` (attrition table printed) |
| 2 | Builds Stage-3 prediction inputs for 3 cohorts x 2 endpoints into **new, separate** `*_noprecastrate_t6mo` trees |
| 3 | Fits the univariate Cox arm on those trees (same runner as `02`) |
| 4-7 | Every table from `02a`: counts/incidence, endpoint overlap, PSA/testosterone pre vs post, univariate associations, the narrowing cascade |
| 8 | Regenerates the PSA/testosterone association forest (PDF) with the new cohorts alongside the six existing ones |

**Prerequisites (already produced by notebook `01`):** the Stage-1 survival cohort, the `metastatic_adt` /
`metastatic_llm` MRN lists, and the Stage-2 `longitudinal_prediction_data_adt.csv`. Existing `01`/`02` trees are
only *read* (as the comparison cohorts) and never modified; missing comparison trees are skipped and reported.

### Definition of the new cohort

`cohort = base cohort  AND  no castrate testosterone before ADT (existing rule)  AND  >= 1 testosterone >= 50 ng/dL in [-183, 0) days`

* **Above castrate** = `LAB_VALUE >= 50` ng/dL (the pipeline's castrate cut is `< 50`). Values are already unit-standardized upstream.
* **6 months** = 183 days (`TESTOSTERONE_WINDOW_DAYS`). **Strictly before** the anchor, as in the existing exclusion: a same-day
  value (`t_lab == 0`) may already reflect the first dose.
* The filter **stacks** on `+noprecastrate`: a patient with a documented non-castrate value in the window but an *earlier* castrate value is still excluded.
* Only finite numeric lab values and time offsets count as documented measurements.
* Patients with **no** testosterone in the window are excluded. This is the dominant source of attrition: read the
  event counts in section 4 as power before reading any association.
* `LANDMARK = 180` as in `02a`. Widen `LANDMARKS` to build/fit more.

In [ ]:
import hashlib
import itertools
import uuid
import json
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

HERE = Path.cwd().resolve()
PIPELINE_DIR = next((p for p in (HERE.parent.parent / "pipeline", HERE / "COMPASS" / "pipeline",
                                 HERE / "pipeline", HERE.parent / "pipeline")
                     if (p / "compass_pipeline.py").is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError("Run from COMPASS/notebooks/testing or the repository root")
sys.path.insert(0, str(PIPELINE_DIR))
import compass_pipeline as cp
import cox_aggregated as ca

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
warnings.filterwarnings("ignore", category=FutureWarning)

# --------------------------------------------------------------------------- analysis cross
ARMS = ["adt"]
ENDPOINTS = ("platinum", "nepc")
COHORTS = ("all", "metastatic_adt", "metastatic_llm")
LANDMARK = 180
LANDMARKS = (LANDMARK,)
ID_COL = "DFCI_MRN"

# --------------------------------------------------------------------------- the new filter
TESTOSTERONE_WINDOW_DAYS = 183                      # ~6 months, strictly before ADT start
ABOVE_CASTRATE_NG_DL = cp.CASTRATE_TESTOSTERONE_NG_DL   # 50.0; documented value must be >= this
RUN_SUFFIX = "_t6mo"                                # appended to the *_noprecastrate run label
NEW_EXCLUSION = "pre_adt_castrate_t6mo"             # exclusion key used by this notebook only

OVERWRITE = False         # True: refit univariate models and replace existing outputs
REBUILD_INPUTS = False    # True: rebuild the new prediction-input trees even if present

cp.FORCE_RERUN = OVERWRITE
cp.BUILD_FULL_FOLLOWUP = False   # univariate needs only the landmark files

DATA_ROOT = Path(cp._PROFILE_OUTPUT_ROOT)
SURVIVAL_ROOT = DATA_ROOT / "survival_analysis"
MRN_DIR = DATA_ROOT / "mrn_lists"
LONGITUDINAL_CSV = DATA_ROOT / "longitudinal_prediction_data_adt.csv"
TABLE_DIR = SURVIVAL_ROOT / "cohort_sensitivity_t6mo"        # aggregate tables written here
FIG_DIR = (Path(os.getenv("COMPASS_FIG_ROOT", "/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS"))
           / "03_univariate" / "cohort_sensitivity_t6mo")

EXCLUDE_LIST_PATH = MRN_DIR / "pre_adt_castrate_or_no_testosterone_6mo_exclusion_mrns.csv"
QUALIFYING_PATH = MRN_DIR / "testosterone_6mo_qualifying_patients.csv"

# Hard prerequisites from notebook 01. Fail early with one clear message.
STAGE1_PATH = cp.cohort_mrn_list_path("all", arm="adt", data_root=DATA_ROOT)
prereqs = {"Stage 1 survival cohort": STAGE1_PATH, "Stage 2 longitudinal labs": LONGITUDINAL_CSV}
for cohort in COHORTS:
    if cohort != "all":
        prereqs[f"cohort MRN list ({cohort})"] = cp.cohort_mrn_list_path(cohort, arm="adt", data_root=DATA_ROOT)
absent = {k: v for k, v in prereqs.items() if not Path(v).exists()}
if absent:
    raise FileNotFoundError("Run notebook 01 (Stages 1-2 and the cohort MRN lists) first:\n  "
                            + "\n  ".join(f"{k}: {v}" for k, v in absent.items()))
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print(f"data root : {DATA_ROOT}\ntables    : {TABLE_DIR}\nfigures   : {FIG_DIR}")

## 1. Derive the new exclusion list

One load of PSA and testosterone from the shared Stage-2 table (PSA is reused for section 5). The Stage-2 table is anchor-level, so `t_lab` is
signed days from ADT start. The attrition table walks the stacked filter from the Stage-1 cohort.

In [ ]:
PSA_LAB = "PSA"
TESTOSTERONE_LAB = cp.TESTOSTERONE_LAB_NAME
CASTRATE_NG_DL = cp.CASTRATE_TESTOSTERONE_NG_DL
LABS_OF_INTEREST = (PSA_LAB, TESTOSTERONE_LAB)


def to_int_ids(series):
    # MRNs round-trip through float/str in several CSVs; one canonical nullable-int form.
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def load_longitudinal_labs(path):
    labs = (
        pl.scan_csv(path, infer_schema_length=0)
        .select("DFCI_MRN", "LAB_NAME", "LAB_VALUE", "t_lab")
        .with_columns(
            pl.col("DFCI_MRN").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False),
            pl.col("LAB_VALUE").cast(pl.Float64, strict=False),
            pl.col("t_lab").cast(pl.Float64, strict=False),
        )
        .filter(pl.col("DFCI_MRN").is_not_null() & pl.col("LAB_NAME").is_in(list(LABS_OF_INTEREST))
                & pl.col("LAB_VALUE").is_finite() & pl.col("t_lab").is_finite())
        .collect().to_pandas()
    )
    labs[ID_COL] = to_int_ids(labs[ID_COL])
    return labs


def derive_testosterone_filter(stage1_mrns, labs, *, window_days, threshold):
    # Returns (qualifying frame, exclusion MRN set, attrition table). All logic lives here so it can be unit-checked.
    if not np.isfinite(window_days) or window_days <= 0 or not np.isfinite(threshold):
        raise ValueError("The window must be positive and the window/threshold must be finite.")
    t = labs.loc[labs["LAB_NAME"].eq(TESTOSTERONE_LAB)].copy()
    for col in ("LAB_VALUE", "t_lab"):
        t[col] = pd.to_numeric(t[col], errors="coerce")
    t = t.loc[np.isfinite(t["LAB_VALUE"]) & np.isfinite(t["t_lab"])]
    pre = t.loc[t["t_lab"].lt(0)]                                   # strictly before ADT start
    in_window = pre.loc[pre["t_lab"].ge(-window_days)]

    any_pre = set(pre[ID_COL].dropna().astype(int))
    castrate_pre = set(pre.loc[pre["LAB_VALUE"].lt(threshold), ID_COL].dropna().astype(int))  # existing rule
    any_window = set(in_window[ID_COL].dropna().astype(int))
    above = in_window.loc[in_window["LAB_VALUE"].ge(threshold)]
    above_window = set(above[ID_COL].dropna().astype(int))

    stage1 = set(stage1_mrns)
    after_existing = stage1 - castrate_pre
    qualifying_ids = after_existing & above_window

    nearest = (above.loc[above[ID_COL].isin(qualifying_ids)]
               .sort_values("t_lab").groupby(ID_COL)
               .agg(NEAREST_ABOVE_CASTRATE_T_NG_DL=("LAB_VALUE", "last"),
                    NEAREST_ABOVE_CASTRATE_T_DAYS_BEFORE=("t_lab", lambda s: -float(s.iloc[-1])),
                    N_ABOVE_CASTRATE_T_IN_WINDOW=("LAB_VALUE", "size"))
               .reset_index().sort_values(ID_COL))
    qualifying = nearest.rename(columns={ID_COL: "DFCI_MRN"})

    attrition = pd.DataFrame([
        ("Stage 1 ADT cohort", len(stage1)),
        ("  with any pre-ADT testosterone", len(stage1 & any_pre)),
        (f"  castrate (<{threshold:g}) result before ADT -> removed by existing +noprecastrate", len(stage1 & castrate_pre)),
        ("after existing +noprecastrate", len(after_existing)),
        (f"  of those: any testosterone in the {window_days}d window", len(after_existing & any_window)),
        (f"  of those: testosterone >= {threshold:g} in the {window_days}d window", len(qualifying_ids)),
        ("FINAL: qualifying cohort (kept)", len(qualifying_ids)),
        ("FINAL: excluded (castrate pre-ADT, or no documented non-castrate T in window)", len(stage1 - qualifying_ids)),
    ], columns=["step", "n_patients"])
    attrition["pct_of_stage1"] = 100 * attrition["n_patients"] / max(len(stage1), 1)
    return qualifying, stage1 - qualifying_ids, attrition


LABS = load_longitudinal_labs(LONGITUDINAL_CSV)
print(f"Longitudinal labs: {len(LABS):,} rows, {LABS[ID_COL].nunique():,} patients")

stage1_mrns = set(to_int_ids(pd.read_csv(STAGE1_PATH, usecols=[ID_COL])[ID_COL]).dropna().astype(int))
qualifying, excluded_ids, filter_attrition = derive_testosterone_filter(
    stage1_mrns, LABS, window_days=TESTOSTERONE_WINDOW_DAYS, threshold=ABOVE_CASTRATE_NG_DL)
if qualifying.empty:
    raise ValueError("No patient has a documented non-castrate testosterone in the window; check the Stage-2 table.")

pd.DataFrame({"DFCI_MRN": sorted(excluded_ids)}).to_csv(EXCLUDE_LIST_PATH, index=False)
qualifying.to_csv(QUALIFYING_PATH, index=False)
filter_attrition.to_csv(TABLE_DIR / "filter_attrition.csv", index=False)
QUALIFYING_IDS = set(qualifying["DFCI_MRN"].astype(int))
EXCLUDED_IDS = set(excluded_ids)

display(filter_attrition.round(1))
print(f"\nexclusion list -> {EXCLUDE_LIST_PATH}\nqualifying     -> {QUALIFYING_PATH}")
display(qualifying[["NEAREST_ABOVE_CASTRATE_T_NG_DL", "NEAREST_ABOVE_CASTRATE_T_DAYS_BEFORE",
                    "N_ABOVE_CASTRATE_T_IN_WINDOW"]].describe().round(1))

## 2. Run definitions

`make_endpoint_runs` is the same factory `01`/`02`/`02a` use, so the six existing trees (`REF_RUNS`) resolve to exactly the
directories they were written to. The new runs are derived from the `+noprecastrate` ones with a new label, new input/output trees,
and the combined exclusion list from section 1 passed as Stage 3's `--exclude-mrns`. Nothing in an existing tree is touched.

In [ ]:
REF_RUNS = cp.make_endpoint_runs(ARMS, endpoints=ENDPOINTS, cohorts=COHORTS, exclusions=("none", "pre_adt_castrate"))

NEW_RUNS = []
for base in (r for r in REF_RUNS if r["exclusion"] == "pre_adt_castrate"):
    label = base["label"] + RUN_SUFFIX
    sfx = cp.endpoint_suffix(base["endpoint"])
    run = dict(
        base,
        label=label,
        title=base["title"] + f" / T>={ABOVE_CASTRATE_NG_DL:g} within {TESTOSTERONE_WINDOW_DAYS}d pre-ADT",
        exclusion=NEW_EXCLUSION,
        exclude_mrns=EXCLUDE_LIST_PATH,
        landmarks=list(LANDMARKS),
        inputs_dir=SURVIVAL_ROOT / f"prediction_inputs_{label}{sfx}",
        output_dir=SURVIVAL_ROOT / f"local_runs_{label}{sfx}",
    )
    run["inputs_dir"].mkdir(parents=True, exist_ok=True)
    run["output_dir"].mkdir(parents=True, exist_ok=True)
    NEW_RUNS.append(run)

# Reference cohorts are compared at the same landmark only.
REF_RUNS = [dict(r, landmarks=list(LANDMARKS)) for r in REF_RUNS]
ALL_RUNS = REF_RUNS + NEW_RUNS
assert not {r["label"] for r in REF_RUNS} & {r["label"] for r in NEW_RUNS}
assert not {str(r["inputs_dir"]) for r in REF_RUNS} & {str(r["inputs_dir"]) for r in NEW_RUNS}, "would overwrite an existing tree"

for r in NEW_RUNS:
    print(f"{r['label']:44s} [{r['endpoint']:8s}] inputs={r['inputs_dir'].name}")

## 3. Build prediction inputs and fit the univariate models

Stage 3 uses the combined exclusion list and each endpoint's own validity gate. Cached inputs are reused only when all required input files exist and recorded provenance matches the filter parameters, landmarks, and SHA-256 hashes of the source data and MRN lists. Missing or outdated provenance triggers a rebuild; `REBUILD_INPUTS = True` also forces one.

Every rebuild invalidates the model-completion record **before** work starts and receives a new build ID. Fits are forced whenever their completion record does not match the current inputs, including after an interrupted rebuild or fit. `OVERWRITE = True` additionally forces fitting even with a matching record. Input eligibility is checked before any model is fitted.


In [ ]:
PROVENANCE_FILENAME = "testosterone_filter_provenance.json"
FIT_PROVENANCE_FILENAME = "testosterone_filter_fit_provenance.json"


def read_provenance(path):
    try:
        return json.loads(Path(path).read_text())
    except (FileNotFoundError, json.JSONDecodeError):
        return None


def file_fingerprint(path, cache):
    path = Path(path).resolve()
    key = str(path)
    if key not in cache:
        digest = hashlib.sha256()
        with path.open("rb") as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
        cache[key] = {"path": key, "sha256": digest.hexdigest()}
    return cache[key]


def input_spec(run, cache):
    return {
        "schema_version": 1,
        "window_days": TESTOSTERONE_WINDOW_DAYS,
        "threshold_ng_dl": ABOVE_CASTRATE_NG_DL,
        "testosterone_lab": TESTOSTERONE_LAB,
        "cohort": run["cohort"], "endpoint": run["endpoint"],
        "anchor_col": run["anchor_col"], "landmarks": list(run["landmarks"]),
        "build_full_followup": cp.BUILD_FULL_FOLLOWUP,
        "sources": {name: file_fingerprint(path, cache) for name, path in {
            "stage1": STAGE1_PATH, "longitudinal": run["input_csv"],
            "restriction": run["restrict_to_mrns"], "exclusion": run["exclude_mrns"],
            "qualifying": QUALIFYING_PATH,
        }.items()},
    }


def inputs_complete(run):
    required = [run["inputs_dir"] / "build_manifest.json"]
    for lm in run["landmarks"]:
        required.extend(run["inputs_dir"] / f"{stem}{lm}.csv" for stem in
                        ("aggregated_landmark", "pre_treatment_lab_long_landmark"))
    return all(path.is_file() for path in required)


def check_input_eligibility(run):
    for lm in run["landmarks"]:
        frame = pd.read_csv(run["inputs_dir"] / f"aggregated_landmark{lm}.csv", usecols=[ID_COL])
        ids = set(to_int_ids(frame[ID_COL]).dropna().astype(int))
        if ids - QUALIFYING_IDS or ids & EXCLUDED_IDS:
            raise ValueError(f"{run['label']} [{run['endpoint']}] landmark {lm}: "
                             "input patients do not satisfy the current filter; rebuild inputs.")


# Fresh hashes on every execution; shared source files are read only once per cell.
source_fingerprints = {}
for run in NEW_RUNS:
    provenance_path = run["inputs_dir"] / PROVENANCE_FILENAME
    fit_path = run["output_dir"] / FIT_PROVENANCE_FILENAME
    expected = input_spec(run, source_fingerprints)
    previous = read_provenance(provenance_path)
    current = (isinstance(previous, dict) and previous.get("spec") == expected
               and bool(previous.get("build_id")))
    if REBUILD_INPUTS or not inputs_complete(run) or not current:
        # Persist invalidation so restarting the kernel cannot reuse an older fit.
        fit_path.unlink(missing_ok=True)
        provenance_path.unlink(missing_ok=True)
        cp.build_prediction_inputs(run)
        if not inputs_complete(run):
            raise RuntimeError(f"Stage 3 did not produce all required files for {run['label']}")
        check_input_eligibility(run)
        provenance = {"spec": expected, "build_id": uuid.uuid4().hex}
        provenance_path.write_text(json.dumps(provenance, indent=2) + "\n")
    else:
        check_input_eligibility(run)
        print(f"[skip] inputs and provenance match: {run['inputs_dir'].name}")


In [ ]:
for run in NEW_RUNS:
    provenance_path = run["inputs_dir"] / PROVENANCE_FILENAME
    fit_path = run["output_dir"] / FIT_PROVENANCE_FILENAME
    provenance = read_provenance(provenance_path)
    if not isinstance(provenance, dict) or not provenance.get("build_id") or not inputs_complete(run):
        raise RuntimeError("Run the prediction-input build cell before fitting.")
    check_input_eligibility(run)
    force_refit = OVERWRITE or read_provenance(fit_path) != provenance
    fit_path.unlink(missing_ok=True)
    previous_force = cp.FORCE_RERUN
    try:
        cp.FORCE_RERUN = force_refit
        summary = cp.run_univariate(run)
        expected_tasks = cp.tasks_for_run(run, cp.UNIVARIATE_TASK_SPECS)
        allowed_statuses = {"ok"} if force_refit else {"ok", "skipped"}
        if len(summary) != len(expected_tasks) or any(status not in allowed_statuses for _, status, _ in summary):
            raise RuntimeError(f"Univariate fitting failed for {run['label']} [{run['endpoint']}]; "
                               "the next run will force a refit.")
        for model, lm, config, filename in expected_tasks:
            result = run["output_dir"] / cp.model_output_dir(model) / f"landmark_{lm}" / config / filename
            if not result.is_file():
                raise RuntimeError(f"Expected univariate output is missing: {result}")
        fit_path.write_text(json.dumps(provenance, indent=2) + "\n")
    finally:
        cp.FORCE_RERUN = previous_force


In [ ]:
NOMINAL_ALPHA = 0.05
for run in NEW_RUNS:
    filtered = cp.filter_nominal(cp.load_univariate_results(run), alpha=NOMINAL_ALPHA)
    export_path = run["output_dir"] / "cox" / "nominally_significant_univariate_results.csv"
    export_path.parent.mkdir(parents=True, exist_ok=True)
    filtered.to_csv(export_path, index=False)
    print(f"{run['label']} [{run['endpoint']}]: {len(filtered)} nominally significant rows -> {export_path}")

### Cohort integrity check

The new trees must contain only qualifying patients, and every patient must by construction have a documented non-castrate testosterone in the window.

In [ ]:
def normalize_id_set(frame):
    return set(to_int_ids(frame[ID_COL]).dropna().astype(int))


integrity = []
for run in NEW_RUNS:
    for lm in run["landmarks"]:
        agg = pd.read_csv(run["inputs_dir"] / f"aggregated_landmark{lm}.csv", usecols=[ID_COL], low_memory=False)
        ids = normalize_id_set(agg)
        integrity.append({"run": run["label"], "endpoint": run["endpoint"], "landmark_days": lm,
                          "n_patients": len(ids),
                          "n_excluded_present": len(ids & EXCLUDED_IDS),
                          "n_not_qualifying": len(ids - QUALIFYING_IDS)})
integrity = pd.DataFrame(integrity)
display(integrity)
assert integrity[["n_excluded_present", "n_not_qualifying"]].to_numpy().sum() == 0, \
    "An excluded patient leaked into a new tree; rerun with REBUILD_INPUTS = True"
print("OK: every patient in every new tree qualifies, none of the excluded patients present.")

## 4. Cohort numbers and event incidence

Shared helpers, then `02a` section 1 across the 9 patient cohorts (3 cohorts x {`none`, `+noprecastrate`, `+noprecastrate +T6mo`}). Missing reference trees are listed at the end, not fatal.

In [ ]:
def cohort_label(run):
    suffix = {"none": "", "pre_adt_castrate": " +noprecastrate", NEW_EXCLUSION: " +noprecastrate +T6mo"}[run["exclusion"]]
    return f"{run['cohort']}{suffix}"


EXCLUSION_ORDER = ("none", "pre_adt_castrate", NEW_EXCLUSION)
COHORT_ORDER = [cohort_label({"cohort": c, "exclusion": e}) for c in COHORTS for e in EXCLUSION_ORDER]
NEW_LABELS = {cohort_label(r) for r in NEW_RUNS}
EVENT_COL = {ep: ca.ENDPOINTS[ep]["event_col"] for ep in ENDPOINTS}
DURATION_COL = {ep: ca.ENDPOINTS[ep]["duration_col"] for ep in ENDPOINTS}
MISSING = []


def order_cohorts(frame, col="cohort"):
    if col not in frame.columns:
        return frame
    out = frame.copy()
    out[col] = pd.Categorical(out[col], categories=COHORT_ORDER, ordered=True)
    sort_cols = [col] + [c for c in ("endpoint", "landmark_days") if c in out.columns]
    return out.sort_values(sort_cols).reset_index(drop=True)


def require(path, what):
    if Path(path).exists():
        return True
    MISSING.append(f"{what}: {path}")
    return False


def agg_path(run, landmark):
    return run["inputs_dir"] / f"aggregated_landmark{landmark}.csv"


def save_table(frame, name):
    frame.to_csv(TABLE_DIR / f"{name}.csv", index=False)
    return frame


# --- 4a. per cohort x endpoint x landmark: what each model actually fit
landmark_rows = []
for run in ALL_RUNS:
    ep = run["endpoint"]
    event_col, duration_col = EVENT_COL[ep], DURATION_COL[ep]
    for lm in LANDMARKS:
        base = {"cohort": cohort_label(run), "endpoint": ep, "landmark_days": lm}
        path = agg_path(run, lm)
        if not require(path, f"{run['label']} [{ep}] aggregated landmark {lm}"):
            landmark_rows.append({**base, "status": "missing"})
            continue
        agg = pd.read_csv(path, low_memory=False)
        if event_col not in agg.columns:
            landmark_rows.append({**base, "status": f"no {event_col} column"})
            continue
        events = pd.to_numeric(agg[event_col], errors="coerce").fillna(0)
        has_duration = duration_col in agg.columns
        durations = pd.to_numeric(agg[duration_col], errors="coerce") if has_duration else None
        landmark_rows.append({
            **base, "n_patients": len(agg), "n_events": int(events.eq(1).sum()),
            "event_rate_pct": 100 * events.eq(1).mean(),
            "median_days_to_event": durations.loc[events.eq(1)].median() if has_duration else np.nan,
            "median_days_followup": durations.median() if has_duration else np.nan,
            "status": "ok" if has_duration else f"ok ({duration_col} absent)",
        })

counts = order_cohorts(pd.DataFrame(landmark_rows))
save_table(counts, "counts_and_incidence")
display(counts.round(1))

In [ ]:
ok_counts = counts.loc[counts["status"].astype(str).str.startswith("ok")]
THIN_EVENTS = 25
if not ok_counts.empty:
    for value, title in (("n_events", "events"), ("n_patients", "patients"), ("event_rate_pct", "event rate (%)")):
        grid = ok_counts.pivot_table(index="cohort", columns=["endpoint", "landmark_days"],
                                     values=value, observed=False).reindex(COHORT_ORDER).dropna(how="all")
        print(f"\n=== {title} ===")
        display(grid.round(1))
    thin = ok_counts.loc[ok_counts["n_events"].lt(THIN_EVENTS)]
    if len(thin):
        print(f"\n!! {len(thin)} cell(s) with < {THIN_EVENTS} events -- underpowered; read metrics there as power, not biology:")
        display(thin[["cohort", "endpoint", "landmark_days", "n_patients", "n_events"]])
    else:
        print(f"\nAll cells have >= {THIN_EVENTS} events.")

### 4c. Endpoint overlap within each patient cohort

Each endpoint's build applies its own incident gate, so platinum and NEPC are not the same patients; this quantifies the gap.

In [ ]:
RUN_BY = {(cohort_label(r), r["endpoint"]): r for r in ALL_RUNS}
overlap_rows = []
for label in COHORT_ORDER:
    for lm in LANDMARKS:
        ids = {}
        for ep in ENDPOINTS:
            run = RUN_BY.get((label, ep))
            if run is None or not agg_path(run, lm).exists():
                continue
            ids[ep] = normalize_id_set(pd.read_csv(agg_path(run, lm), usecols=[ID_COL], low_memory=False))
        if len(ids) < 2:
            continue
        a, b = ENDPOINTS[:2]
        sa, sb = ids[a], ids[b]
        overlap_rows.append({"cohort": label, "landmark_days": lm, f"n_{a}": len(sa), f"n_{b}": len(sb),
                             "n_shared": len(sa & sb), f"n_{a}_only": len(sa - sb), f"n_{b}_only": len(sb - sa),
                             f"pct_of_{a}_retained": 100 * len(sa & sb) / len(sa) if sa else np.nan})
if overlap_rows:
    display(save_table(order_cohorts(pd.DataFrame(overlap_rows)), "endpoint_overlap").round(1))

## 5. PSA and testosterone: pre- vs post-treatment, stratified by event

Same definitions as `02a` section 2. The pre/post split stays at the ADT anchor (`t_lab < 0` vs `1..365`) even though the cohort is the +180d one.
`PSA` is the narrow OMOP-collapsed `LAB_NAME == "PSA"` series. **Sanity read for the new cohort:** its pre-treatment testosterone must show a non-castrate
median and a lower castrate fraction than the cohorts without the requirement.

In [ ]:
from scipy import stats as sps

POST_WINDOW = (1, 365)
PRE_POST_LANDMARK = LANDMARK
PRE_POST_BOUNDARY = 0


def summarize_lab(values, lab_name):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    if values.empty:
        return {}
    out = {"n_patients": int(len(values)), "median": float(values.median()), "q1": float(values.quantile(0.25)),
           "q3": float(values.quantile(0.75)), "mean": float(values.mean())}
    if lab_name == PSA_LAB:
        out["mean_log1p"] = float(np.log1p(values.clip(lower=0)).mean())
    if lab_name == TESTOSTERONE_LAB:
        out["pct_castrate"] = float(100 * values.lt(CASTRATE_NG_DL).mean())
    return out


def per_patient_lab(labs, mrns, lab_name, window):
    # One value per patient (median of results in the window) so a patient with 40 draws does not outweigh one with a single draw.
    lo, hi = window
    sub = labs.loc[labs["LAB_NAME"].eq(lab_name) & labs[ID_COL].isin(mrns) & labs["t_lab"].ge(lo) & labs["t_lab"].le(hi)]
    return sub.groupby(ID_COL)["LAB_VALUE"].median() if len(sub) else pd.Series(dtype=float)


def load_run_cohort(run, landmark):
    path = agg_path(run, landmark)
    event_col = EVENT_COL[run["endpoint"]]
    if not path.exists():
        return None
    agg = pd.read_csv(path, low_memory=False)
    if event_col not in agg.columns or ID_COL not in agg.columns:
        return None
    agg[ID_COL] = to_int_ids(agg[ID_COL])
    agg = agg.dropna(subset=[ID_COL])
    return agg[ID_COL].astype(int).to_numpy(), pd.to_numeric(agg[event_col], errors="coerce").fillna(0).to_numpy()


PRE_WINDOW = (-np.inf, PRE_POST_BOUNDARY - 1e-9)
POST_WINDOW_EFF = (max(POST_WINDOW[0], PRE_POST_BOUNDARY), POST_WINDOW[1])

prepost_rows = []
for run in ALL_RUNS:
    loaded = load_run_cohort(run, PRE_POST_LANDMARK)
    if loaded is None:
        continue
    ids, events = loaded
    for lab in LABS_OF_INTEREST:
        for ev in (0, 1):
            mrns = set(ids[events == ev].tolist())
            if not mrns:
                continue
            for period, window in (("pre", PRE_WINDOW), ("post", POST_WINDOW_EFF)):
                stats = summarize_lab(per_patient_lab(LABS, mrns, lab, window), lab)
                if stats:
                    prepost_rows.append({"cohort": cohort_label(run), "endpoint": run["endpoint"], "lab": lab,
                                         "period": period, "event": ev, "n_cohort": len(mrns), **stats})
prepost = pd.DataFrame(prepost_rows)
if len(prepost):
    prepost = save_table(order_cohorts(prepost), "pre_post_lab_summary")
    print(f"Cohort: landmark +{PRE_POST_LANDMARK}d | pre: t_lab < {PRE_POST_BOUNDARY}d | post: {POST_WINDOW_EFF[0]}-{POST_WINDOW_EFF[1]}d from ADT start\n")
    display(prepost.round(2))
else:
    print("No pre/post lab summaries could be built; see missing artifacts.")

In [ ]:
if len(prepost):
    for lab in LABS_OF_INTEREST:
        grid = (prepost.loc[prepost["lab"].eq(lab)]
                .pivot_table(index=["cohort", "endpoint"], columns=["period", "event"], values="median", observed=False))
        grid = grid.reindex(columns=pd.MultiIndex.from_product([["pre", "post"], [0, 1]], names=["period", "event"]),
                            fill_value=np.nan).dropna(how="all")
        print(f"\n=== {lab}: median per-patient value, by event status ===")
        display(grid.round(2))
    t_sub = prepost.loc[prepost["lab"].eq(TESTOSTERONE_LAB)]
    if len(t_sub):
        grid = t_sub.pivot_table(index=["cohort", "endpoint"], columns=["period", "event"], values="pct_castrate", observed=False)
        grid = grid.reindex(columns=pd.MultiIndex.from_product([["pre", "post"], [0, 1]], names=["period", "event"]),
                            fill_value=np.nan).dropna(how="all")
        print(f"\n=== Testosterone: % of patients below {CASTRATE_NG_DL:g} ng/dL (per-patient median) ===")
        display(grid.round(1))

### 5b. Within-patient pre -> post change

Patients with both a pre- and a post-treatment value, paired with themselves; Mann-Whitney on the change, event=1 vs event=0 (unadjusted, descriptive).
Note that the new filter mechanically guarantees every patient a pre-treatment testosterone, so the paired testosterone set is larger here than in the unfiltered cohorts.

In [ ]:
delta_rows = []
for run in ALL_RUNS:
    loaded = load_run_cohort(run, PRE_POST_LANDMARK)
    if loaded is None:
        continue
    ids, events = loaded
    event_by_mrn = pd.Series(events, index=ids)
    event_by_mrn = event_by_mrn[~event_by_mrn.index.duplicated()]
    all_mrns = set(ids.tolist())
    for lab in LABS_OF_INTEREST:
        pre = per_patient_lab(LABS, all_mrns, lab, PRE_WINDOW)
        post = per_patient_lab(LABS, all_mrns, lab, POST_WINDOW_EFF)
        paired = pd.DataFrame({"pre": pre, "post": post}).dropna()
        if paired.empty:
            continue
        paired["event"] = paired.index.map(event_by_mrn)
        paired = paired.dropna(subset=["event"])
        if lab == PSA_LAB:
            paired["delta"] = np.log1p(paired["post"].clip(lower=0)) - np.log1p(paired["pre"].clip(lower=0))
            units = "log1p(ng/mL)"
        else:
            paired["delta"] = paired["post"] - paired["pre"]
            units = "ng/dL"
        groups = {ev: g["delta"].dropna() for ev, g in paired.groupby("event")}
        row = {"cohort": cohort_label(run), "endpoint": run["endpoint"], "lab": lab,
               "delta_units": units, "n_paired": int(len(paired))}
        for ev in (0, 1):
            g = groups.get(ev, pd.Series(dtype=float))
            row[f"n_event{ev}"] = int(len(g))
            row[f"median_delta_event{ev}"] = float(g.median()) if len(g) else np.nan
        g0, g1 = groups.get(0, pd.Series(dtype=float)), groups.get(1, pd.Series(dtype=float))
        row["p_delta_differs"] = (float(sps.mannwhitneyu(g0, g1, alternative="two-sided").pvalue)
                                  if len(g0) >= 3 and len(g1) >= 3 else np.nan)
        delta_rows.append(row)
deltas = pd.DataFrame(delta_rows)
if len(deltas):
    display(save_table(order_cohorts(deltas), "pre_post_paired_change").round(3))
else:
    print("No patients had both a pre- and a post-treatment value.")

## 6. Univariate associations across endpoints and cohorts

Hazard ratios per SD from `cox_agg_univariate_nobs_adjusted.csv`, loaded with `cp.load_univariate_results` for every available run. `n_observations`
rows are dropped as a surveillance-intensity proxy (as in `02a`). **Read against section 4:** the confidence intervals are kept in every table because the
new cohort's event counts, not its point estimates, determine what can be concluded.

In [ ]:
HR_COL = "hazard_ratio_per_sd"
frames = []
for run in ALL_RUNS:
    try:
        frame = cp.load_univariate_results(run)
    except (FileNotFoundError, ValueError) as exc:
        MISSING.append(f"{run['label']} [{run['endpoint']}] univariate: {exc}")
        continue
    frame["run_label"] = run["label"]
    frame["cohort"] = cohort_label(run)
    frame["endpoint"] = run["endpoint"]
    frames.append(frame)

univariate = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(univariate):
    for col in (HR_COL, "ci_lower", "ci_upper", "p_value", "q_value"):
        if col in univariate.columns:
            univariate[col] = pd.to_numeric(univariate[col], errors="coerce")
    stat = (univariate["feature_stat"] if "feature_stat" in univariate.columns
            else univariate["feature"].astype(str).str.split("__").str[-1])
    n_dropped = int(stat.astype(str).eq("n_observations").sum())
    univariate = univariate.loc[~stat.astype(str).eq("n_observations")].copy()
    print(f"Dropped {n_dropped:,} n_observations row(s). Univariate rows: {len(univariate):,} across "
          f"{univariate['cohort'].nunique()} cohorts x {univariate['endpoint'].nunique()} endpoints")
else:
    print("No univariate results found.")

In [ ]:
def is_lab_of_interest(frame):
    name = (frame["lab_name"] if "lab_name" in frame.columns else frame["feature"].astype(str).str.split("__").str[0]).astype(str).str.lower()
    return name.str.startswith("psa") | name.str.contains("prostate specific ag") | name.str.contains("testosterone")


lab_univariate = pd.DataFrame()
UNIVARIATE_LANDMARK = LANDMARK
if len(univariate):
    lab_univariate = univariate.loc[is_lab_of_interest(univariate)].copy()
    keep = [c for c in ["cohort", "endpoint", "landmark_days", "feature", "lab_name", "feature_stat", "n_patients_used",
                        "n_events_used", HR_COL, "ci_lower", "ci_upper", "p_value", "q_value"] if c in lab_univariate.columns]
    lab_univariate = order_cohorts(lab_univariate[keep])
    save_table(lab_univariate, "psa_testosterone_univariate")
    display(lab_univariate.sort_values(["cohort", "endpoint", "landmark_days", "p_value"]).round(4))

In [ ]:
if len(lab_univariate):
    at_lm = lab_univariate.loc[lab_univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)]
    for value, title in ((HR_COL, "hazard ratio per SD"), ("p_value", "p-value")):
        grid = at_lm.pivot_table(index="feature", columns=["endpoint", "cohort"], values=value, observed=False)
        print(f"\n=== {title} | landmark +{UNIVARIATE_LANDMARK}d ===")
        display(grid.round(4))
    sig = (at_lm.assign(nominal=at_lm["p_value"].lt(0.05)).groupby(["feature", "endpoint"], observed=False)["nominal"]
           .agg(n_cohorts_nominal="sum", n_cohorts="size").reset_index())
    print(f"\n=== nominal significance (p < 0.05) across cohorts | landmark +{UNIVARIATE_LANDMARK}d ===")
    display(sig.sort_values(["endpoint", "n_cohorts_nominal"], ascending=[True, False]))

In [ ]:
if len(univariate):
    at_lm = univariate.loc[univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)].copy()
    TOP_N = 8
    top = at_lm.dropna(subset=["p_value"]).sort_values("p_value").groupby(["cohort", "endpoint"], observed=False).head(TOP_N)
    cols = [c for c in ["cohort", "endpoint", "feature", "n_events_used", HR_COL, "ci_lower", "ci_upper", "p_value", "q_value"] if c in top.columns]
    print(f"Top {TOP_N} features by p-value per cell | landmark +{UNIVARIATE_LANDMARK}d")
    display(save_table(order_cohorts(top[cols]), "top_features_per_cell").round(4))

    summary = at_lm.groupby(["cohort", "endpoint"], observed=False).apply(
        lambda g: pd.Series({
            "n_features": len(g), "n_nominal_p05": int(g["p_value"].lt(0.05).sum()),
            "n_fdr_q05": int(g["q_value"].lt(0.05).sum()) if "q_value" in g.columns else np.nan,
            "min_p": g["p_value"].min(),
            "median_n_events_used": g["n_events_used"].median() if "n_events_used" in g.columns else np.nan,
        }), include_groups=False).reset_index()
    print("\n=== association yield per cell ===")
    display(save_table(order_cohorts(summary), "association_yield").round(4))

## 7. Effect of the new filter

Three steps are isolated within each base cohort, because shrinking N must never be read as a real effect:
`none -> +noprecastrate` (existing), `+noprecastrate -> +T6mo` (this notebook's marginal filter), and `none -> +T6mo` (total).

In [ ]:
ok = counts.loc[counts["status"].astype(str).str.startswith("ok")].copy()
ok["cohort"] = ok["cohort"].astype(str)
idx = ok.set_index(["cohort", "endpoint", "landmark_days"])


def get(label, ep, lm):
    try:
        return idx.loc[(label, ep, lm)]
    except KeyError:
        return None


step_rows = []
for ep in ENDPOINTS:
    for lm in LANDMARKS:
        for coh in COHORTS:
            none, pca, t6 = get(coh, ep, lm), get(f"{coh} +noprecastrate", ep, lm), get(f"{coh} +noprecastrate +T6mo", ep, lm)
            for name, before, after in ((f"{coh}: none -> +noprecastrate", none, pca),
                                        (f"{coh}: +noprecastrate -> +T6mo", pca, t6),
                                        (f"{coh}: none -> +T6mo (total)", none, t6)):
                if before is None or after is None:
                    continue
                step_rows.append({
                    "step": name, "endpoint": ep, "landmark_days": lm,
                    "n_before": before["n_patients"], "n_after": after["n_patients"],
                    "pct_patients_retained": 100 * after["n_patients"] / before["n_patients"] if before["n_patients"] else np.nan,
                    "events_before": before["n_events"], "events_after": after["n_events"],
                    "pct_events_retained": 100 * after["n_events"] / before["n_events"] if before["n_events"] else np.nan,
                    "rate_before_pct": before["event_rate_pct"], "rate_after_pct": after["event_rate_pct"],
                    "rate_ratio": after["event_rate_pct"] / before["event_rate_pct"] if before["event_rate_pct"] else np.nan,
                })
steps = pd.DataFrame(step_rows)
if len(steps):
    display(save_table(steps, "narrowing_steps").round(2))
else:
    print("Cohort counts are needed for the narrowing table; see section 4.")

In [ ]:
# Does the PSA/testosterone association survive the narrowing? Same feature down the cascade.
if len(lab_univariate):
    at_lm = lab_univariate.loc[lab_univariate["landmark_days"].eq(UNIVARIATE_LANDMARK)]
    for ep in ENDPOINTS:
        block = at_lm.loc[at_lm["endpoint"].eq(ep)]
        if block.empty:
            continue
        wide = block.pivot_table(index="feature", columns="cohort", values=[HR_COL, "p_value", "n_events_used"], observed=False)
        wide = wide.reindex(columns=pd.MultiIndex.from_product([[HR_COL, "p_value", "n_events_used"], COHORT_ORDER]),
                            fill_value=np.nan).dropna(how="all").dropna(axis=1, how="all")
        print(f"\n=== {ep}: PSA/testosterone across the cascade | landmark +{UNIVARIATE_LANDMARK}d ===")
        display(wide.round(4))

## 8. PSA/testosterone association figure

The Figure 3 cohort-sensitivity association panel, regenerated with the new cohorts added. Layout and encoding follow the manuscript panel
(`manuscript_dfci_cohort_sensitivity`): rows are analytes oriented so that **HR > 1 always means greater hazard in the labelled direction**
(higher PSA, *lower* testosterone: the testosterone estimates and CIs are inverted), columns are the four summary statistics, one row per cohort,
log hazard-ratio axis with a dashed null at 1, filled points for q < 0.05 and open points otherwise. Cohorts added here are shaded.
Output is PDF only.

In [ ]:
import textwrap
import matplotlib
matplotlib.use("Agg")  # PDF files only: no inline PNG/SVG rendering
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

matplotlib.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif",
                            "axes.spines.top": False, "axes.spines.right": False})

STAT_ORDER = ["mean", "min", "max", "last"]
STAT_LABEL = {"mean": "Mean", "min": "Minimum", "max": "Maximum", "last": "Last"}
ANALYTE_LABEL = {"PSA": "Higher PSA", "Testosterone": "Lower Testosterone"}
ANALYTE_COLOR = {"PSA": "#0072B2", "Testosterone": "#D55E00"}
BASE_LABEL = {"all": "All ADT", "metastatic_adt": "Metastatic: ADT intent", "metastatic_llm": "Metastatic: LLM"}
SUFFIX_LABEL = {"none": "", "pre_adt_castrate": "; no pre-ADT castrate",
                NEW_EXCLUSION: f"; no pre-ADT castrate; T ≥{ABOVE_CASTRATE_NG_DL:g} ng/dL\nwithin {TESTOSTERONE_WINDOW_DAYS} d pre-ADT"}
FIGURE_LABEL = {cohort_label({"cohort": c, "exclusion": e}): BASE_LABEL[c] + SUFFIX_LABEL[e]
                for c in COHORTS for e in EXCLUSION_ORDER}


def prepare_forest(frame, endpoint, landmark):
    # PSA / testosterone x mean/min/max/last rows; testosterone inverted so HR > 1 = greater hazard with lower values.
    d = frame.loc[frame["endpoint"].astype(str).str.lower().eq(endpoint) & frame["landmark_days"].eq(landmark)].copy()
    lab = (d["lab_name"] if "lab_name" in d.columns else d["feature"].astype(str).str.split("__").str[0]).astype(str).str.lower()
    d["analyte"] = np.where(lab.eq("psa") | lab.str.contains("prostate specific ag", regex=False), "PSA",
                            np.where(lab.str.contains("testosterone", regex=False), "Testosterone", None))
    st = (d["feature_stat"] if "feature_stat" in d.columns else d["feature"].astype(str).str.split("__").str[-1]).astype(str).str.lower()
    d["stat"] = st
    d = d.loc[d["analyte"].notna() & d["stat"].isin(STAT_ORDER)].copy()
    for c in (HR_COL, "ci_lower", "ci_upper"):
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.loc[np.isfinite(d[HR_COL]) & d[HR_COL].gt(0)].copy()
    for c in ("ci_lower", "ci_upper"):
        d.loc[~np.isfinite(d[c]) | d[c].le(0), c] = np.nan
    if d.duplicated(["cohort", "analyte", "stat"]).any():
        raise ValueError("Duplicate cohort/analyte/statistic estimates in the forest input")
    low = d["analyte"].eq("Testosterone")
    lo, hi = 1 / d.loc[low, "ci_upper"], 1 / d.loc[low, "ci_lower"]
    d.loc[low, HR_COL] = 1 / d.loc[low, HR_COL]
    d.loc[low, "ci_lower"], d.loc[low, "ci_upper"] = lo, hi
    d["significant"] = d["q_value"].lt(0.05) if "q_value" in d.columns else False
    return d


def plot_forest(d, endpoint, landmark):
    cohorts = [c for c in COHORT_ORDER if c in set(d["cohort"])]
    ypos = {c: len(cohorts) - 1 - i for i, c in enumerate(cohorts)}
    vals = d[[HR_COL, "ci_lower", "ci_upper"]].to_numpy().ravel()
    vals = vals[np.isfinite(vals) & (vals > 0)]
    lo, hi = np.log(min(vals.min(), 1)), np.log(max(vals.max(), 1))
    pad = max((hi - lo) * 0.06, 0.05)
    xlim = (np.exp(lo - pad), np.exp(hi + pad))

    fig, axes = plt.subplots(2, 4, sharex=True, sharey=True, figsize=(12.5, max(4.2, 0.62 * len(cohorts) * 2 + 1.8)))
    for r, analyte in enumerate(("PSA", "Testosterone")):
        for k, stat in enumerate(STAT_ORDER):
            ax = axes[r, k]
            for c in cohorts:                                   # shade the cohorts added by this notebook
                if c in NEW_LABELS:
                    ax.axhspan(ypos[c] - 0.5, ypos[c] + 0.5, color="#f3efe3", zorder=0, lw=0)
            ax.axvline(1, ls="--", lw=0.8, color="#8c8c8c", zorder=1)
            sub = d.loc[d["analyte"].eq(analyte) & d["stat"].eq(stat)]
            for _, row in sub.iterrows():
                y = ypos[row["cohort"]]
                if np.isfinite(row["ci_lower"]) and np.isfinite(row["ci_upper"]):
                    ax.plot([row["ci_lower"], row["ci_upper"]], [y, y], color=ANALYTE_COLOR[analyte], lw=1.4, zorder=2)
                ax.plot(row[HR_COL], y, "o", ms=6, mew=1.4, color=ANALYTE_COLOR[analyte],
                        mfc=ANALYTE_COLOR[analyte] if row["significant"] else "white", zorder=3)
            ax.set_xscale("log")
            ax.set_xlim(*xlim)
            ax.set_ylim(-0.7, len(cohorts) - 0.3)
            ax.set_yticks([ypos[c] for c in cohorts])
            ax.set_yticklabels(["\n".join(textwrap.wrap(FIGURE_LABEL[c].replace("\n", " "), 30)) for c in cohorts], fontsize=7.5)
            ax.tick_params(axis="x", labelsize=8)
            ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
            ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
            if r == 0:
                ax.set_title(STAT_LABEL[stat], fontsize=10, fontweight="bold")
            if k == 0:
                ax.set_ylabel(ANALYTE_LABEL[analyte], fontsize=10, fontweight="bold")
    for ax in axes[1]:
        ax.set_xlabel("Hazard ratio per SD (95% CI; log scale)", fontsize=8.5)
    handles = [Line2D([], [], marker="o", ls="", color="#444", mfc="#444", label="q < 0.05"),
               Line2D([], [], marker="o", ls="", color="#444", mfc="white", label="q ≥ 0.05 or unavailable"),
               matplotlib.patches.Patch(color="#f3efe3", label=f"added: T ≥{ABOVE_CASTRATE_NG_DL:g} ng/dL documented within {TESTOSTERONE_WINDOW_DAYS} d pre-ADT")]
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8.5)
    fig.suptitle(f"PSA and testosterone associations with {endpoint.upper()}  |  treatment landmark +{landmark} days",
                 x=0.01, ha="left", fontsize=11.5, fontweight="bold")
    fig.text(0.01, 0.005, "Testosterone estimates inverted so ratios above 1 mean greater hazard with lower values. Cohorts overlap; estimates are descriptive.",
             fontsize=7.5, ha="left", va="bottom", color="#555")
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))
    return fig


FIG_DIR.mkdir(parents=True, exist_ok=True)
forest_data = {}
if len(univariate):
    for ep in ENDPOINTS:
        d = prepare_forest(univariate, ep, LANDMARK)
        if d.empty:
            print(f"{ep}: no PSA/testosterone estimates available")
            continue
        forest_data[ep] = d
        save_table(d.drop(columns=["source_path"], errors="ignore"), f"forest_source_{ep}_landmark{LANDMARK}")
        fig = plot_forest(d, ep, LANDMARK)
        out = FIG_DIR / f"cohort_forest_{ep}_landmark{LANDMARK}_t6mo.pdf"
        fig.savefig(out)          # PDF only
        print(f"wrote {out}")
        plt.close(fig)

## Missing artifacts

In [ ]:
if MISSING:
    print(f"{len(MISSING)} artifact(s) were not found:")
    for item in dict.fromkeys(MISSING):
        print(f"  {item}")
    print("\nMissing reference trees (the six existing cohorts) only remove those rows from the comparison; "
          "the new cohorts are built and fitted by this notebook.")
else:
    print("All expected artifacts were found.")